## Chapter 6 Ordinary Differential Equation (ODE)

* 6.1 Initial Value Problems
  * 6.1.1 Euler's Method
  * 6.1.2 Existence, uniqueness, and continuity for solutions
  * 6.1.3 First-order linear equations
* 6.2 Analysis of IVP Solvers
  * The Explicit Trapezoid Method
* 6.3 Systems of ODEs
* 6.4 Runge-Kutta Methods and Applications

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import rc

plt.rcParams['xtick.labelsize']=20      # change the tick label size for x axis
plt.rcParams['ytick.labelsize']=20      # change the tick label size for x axis
plt.rcParams['axes.linewidth']=3        # change the line width of the axis
plt.rcParams['xtick.major.width'] = 3   # change the tick line width of x axis
plt.rcParams['ytick.major.width'] = 3   # change the tick line width of y axis
rc('text', usetex=False)                # disable LaTeX rendering in plots
rc('font',**{'family':'DejaVu Sans'})   # set the font of the plot to be DejaVu Sans

### Differential Equation

A differential equation is an equation involving derivatives.  For example,
$$ y'(t) = f(t, y(t))$$
is a first-order differential equation that expresses the rate of change of a quantity $y$ in terms of the present time and the current value of the quantity.

**Why do we care about ODEs?**
1. We can use ODEs to solve projectile motion

[//]: ![Projectile](https://upload.wikimedia.org/wikipedia/commons/a/a4/Parabolic_trajectory.svg)

<img src="https://upload.wikimedia.org/wikipedia/commons/a/a4/Parabolic_trajectory.svg" width="350"/>

2. We can use ODEs to solve infectious disease models

[//]: ![SIR](https://upload.wikimedia.org/wikipedia/commons/2/2a/Graph_SIR_model_without_vital_dynamics.svg)

<img src="https://upload.wikimedia.org/wikipedia/commons/2/2a/Graph_SIR_model_without_vital_dynamics.svg" width="350"/>

3. We can use ODEs to solve many body interactions

[//]: ![three-body](https://upload.wikimedia.org/wikipedia/commons/1/1c/Three-body_Problem_Animation_with_COM.gif)

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1c/Three-body_Problem_Animation_with_COM.gif" width="350"/>

Many times we cannot solve ODEs analytically, so people developed different methods to solve ODEs numerically.

### 6.1 Initial Value Problems

One simple ODE one can solve is the **logistic equation**, which models the rate of change of a population as
$$y'=cy(1-y)$$

Here $y'$ denotes the derivative with respect to time $t$.  

This equation has infinitely many solutions $y(t)$ and by specifying an intial condition, we can identify which of the infinite family of solutions we are interested in.

An **initial value problem** for a first-order ordinary differential equation is the equation together with an initial condition on a specific interval $a\leq t\leq b$:
$$
\left\{
\begin{matrix}
y'=f(t, y)\\
y(a)=y_a\\
t \text{ in } [a, b]
\end{matrix}
\right.
$$

It will be helpful to think of a differential equation as a field of slopes.

Here, our differential equation is
$$
\left\{
\begin{matrix}
y'=cy(1-y)\\
y(a)=y_a\\
t \text{ in } [a, b]
\end{matrix}
\right.
$$


In [ ]:
c = 3
nx, ny = (22, 32)
x = np.linspace(0.01, 2.1, nx)
y = np.linspace(-1, 2.1, ny)
xv, yv = np.meshgrid(x, y)
U = xv
V = c*yv*(1-yv)
U = 10*U / np.sqrt(U**2 + V**2)
V = 10*V / np.sqrt(U**2 + V**2)

In [ ]:
fig = plt.figure(figsize=(6, 8))
plt.rcParams['axes.linewidth']=1
ax = fig.add_subplot(1, 1, 1)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.quiver(xv, yv, U, V, color = '#57829c')
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)

We can solve the **logistic equation** analytically, and as long as the initial condition satisfies $y_0\neq1$, we have
$$y(t)=1-\frac{1}{1+\frac{y_0}{1-y_0}e^{ct}}$$

Using this solution, we can pick two initial conditions ($y_0=0.2,\; y_0=1.3$) and draw them on top of the slope field plot.

In [ ]:
y_0_1 = 0.2
y_0_2 = 2.5
Y_1 = 1 - 1/(1+y_0_1*np.exp(c*x)/(1-y_0_1))
Y_2 = 1 - 1/(1+y_0_2*np.exp(c*x)/(1-y_0_2))

In [ ]:
fig = plt.figure(figsize=(6, 8))
plt.rcParams['axes.linewidth']=1
ax = fig.add_subplot(1, 1, 1)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.quiver(xv, yv, U, V, color = '#57829c')
plt.plot(x, Y_1, linewidth = 2)
plt.plot(x, Y_2, linewidth = 2)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)

### 6.1.1 Euler's Method

As mentioned before, most ODEs don't have explicit solutions like the logistic equation.  By looking at the slope field figure above, one might be able to think of a different approach: to computationally "solve" the differential equation by following arrows.

We want to do the following:
1. Start at initial condition $(t_0, y_0)$
2. Move a short distance
3. Re-evaluate the slope at the new point $(t_1, y_1)$
4. Repeat steps 2 and 3

#### Let's look at an example:
$$
\left\{
\begin{matrix}
y'=ty+t^3\\
y(0)=y_0\\
t \text{ in } [0, 1]
\end{matrix}
\right.
$$

The exact solution for this ODE is
$$y(t)=3e^{t^2/2}-t^2-2$$
for $y(0)=1$.

In [ ]:
nx, ny = (12, 22)
x = np.linspace(0.01, 1.1, nx)
y = np.linspace(0.01, 2.1, ny)
xv, yv = np.meshgrid(x, y)
U = xv
V = xv*yv+xv**3
U = 10*U / np.sqrt(U**2 + V**2)
V = 10*V / np.sqrt(U**2 + V**2)
y_0 = 1.0
Y = 3*np.exp(x**2/2)-x**2-2

In [ ]:
fig = plt.figure(figsize=(6, 8))
plt.rcParams['axes.linewidth']=1
ax = fig.add_subplot(1, 1, 1)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.quiver(xv, yv, U, V, color = '#57829c')
plt.plot(x, Y, linewidth = 2)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)

To perform the Euler's Method, we can create a grid of $n+1$ points
$$t_0<t_1<t_2<\cdots<t_n$$ along the $t$-axis with equal step size $h$.

Using the previous ODE, let's try an array of $t$ values:
$$ t_0=0.0\qquad t_1=0.2\qquad t_2=0.4\qquad t_3=0.6\qquad t_4=0.8\qquad t_5=1.0\qquad$$
with step size $h=0.2$



In [ ]:
y_0 = 1.0
Y = 3*np.exp(x**2/2)-x**2-2

t = np.linspace(0, 1, 6)
print(t)
w = np.zeros(6)
h = 0.2
w[0] = y_0
for i in range(0, len(t)-1):
  w[i+1] = w[i]+h*(t[i]*w[i]+t[i]**3)

# if we have y'=f(t,y)
# Euler's method: y_{t+1}=y_t+h*f(t,y_t)

fig = plt.figure(figsize=(6, 6))
plt.rcParams['axes.linewidth']=1
ax = fig.add_subplot(1, 1, 1)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position(('axes', 0.0))
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.plot(x, Y, linewidth = 2)
plt.plot(t, w, linewidth = 2)
plt.scatter(t, w, s = 50)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)

Being with $w_0=y_0$, following the slope filed at each $t_i$ yields the approximation
$$ w_{i+1}=w_i+hf(t_i, w_i)$$

Thus, we can write Euler's Method as follows:

#### Euler's Method
$$
\begin{align}
w_0&=y_0 \\
w_{i+1}&=w_i+hf(t_i,w_i)
\end{align}
$$

Now let's apply Euler's Method to the example above.

The right-hand side of the differential equation is $f(t,y)=ty+t^3$, so we can write the Euler's Method iteration as
$$
\begin{align}
w_0&=1 \\
w_{i+1}&=w_i+h(t_iw_i+t_i^3)
\end{align}
$$

We can compare the numerical solution and the exact solution with our grid size $h=0.2$.

In [ ]:
h = 0.2
t = np.linspace(0, 1, 6)
y_0 = 1.0
Y = 3*np.exp(t**2/2)-t**2-2

print("step ", "t_i    ",  "y_i     ", "w_i     ", "e_i")
w = np.zeros(6)
w[0] = y_0
for i in range(0, len(t)-1):
  w[i+1] = w[i]+h*(t[i]*w[i]+t[i]**3)
  print(" ", i, " ", "%.1f"%t[i], " ", "%.4f"%w[i], " ", "%.4f"%Y[i], " ", "%.4f"%np.abs(Y[i]-w[i]))
print(" ", 5, " ", "%.1f"%t[5], " ", "%.4f"%w[5], " ", "%.4f"%Y[5], " ", "%.4f"%np.abs(Y[5]-w[5]))

The last column of the table shows the error between Euler's Method and the exact solution, we see that the error grows as we keep moving forward in time.

We can also explore the error behaviors by using a smaller $h$.

In [ ]:
h = 0.1
t = np.linspace(0, 1, 11)
y_0 = 1.0
Y = 3*np.exp(t**2/2)-t**2-2

print("step ", "t_i    ",  "y_i     ", "w_i     ", "e_i")
w = np.zeros(11)
w[0] = y_0
for i in range(0, len(t)-1):
  w[i+1] = w[i]+h*(t[i]*w[i]+t[i]**3)
  print(" ", i, " ", "%.1f"%t[i], " ", "%.4f"%w[i], " ", "%.4f"%Y[i], " ", "%.4f"%np.abs(Y[i]-w[i]))
print(" ", i+1, "", "%.1f"%t[i+1], " ", "%.4f"%w[i+1], " ", "%.4f"%Y[i+1], " ", "%.4f"%np.abs(Y[i+1]-w[i+1]))

We can see that, by cutting $h$ in half, we also cut the error at $t=1.0$ approximately in half.  See plots below for solutions of $h=0.1$ and $h=0.05%=$.

In [ ]:
y_0 = 1.0

fig = plt.figure(figsize=(12, 6))
plt.rcParams['axes.linewidth']=1

t = np.linspace(0, 1, 11)
Y = 3*np.exp(t**2/2)-t**2-2
w = np.zeros(11)
h = 0.1
w[0] = y_0
for i in range(0, len(t)-1):
  w[i+1] = w[i]+h*(t[i]*w[i]+t[i]**3)

ax = fig.add_subplot(1, 2, 1)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position(('axes', 0.0))
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.plot(t, Y, linewidth = 2)
plt.plot(t, w, linewidth = 2)
plt.scatter(t, w, s = 50)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)
plt.title("h=0.1", fontsize = 25)

t = np.linspace(0, 1, 21)
Y = 3*np.exp(t**2/2)-t**2-2
w = np.zeros(21)
h = 0.05
w[0] = y_0
for i in range(0, len(t)-1):
  w[i+1] = w[i]+h*(t[i]*w[i]+t[i]**3)

ax = fig.add_subplot(1, 2, 2)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position(('axes', 0.0))
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.plot(t, Y, linewidth = 2)
plt.plot(t, w, linewidth = 2)
plt.scatter(t, w, s = 50)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)
plt.title("h=0.05", fontsize = 25)

#### Pen and Paper Exercise
Find the Euler’s Method formula for the following initial value problem:
$$
\left\{
\begin{matrix}
y'=cy\\
y(0)=y_0\\
t \text{ in } [0, 1]
\end{matrix}
\right.
$$

In [ ]:
t = np.linspace(0, 1, 11)
w = np.zeros(11)
h = 0.1
y_0 = 1.0
c = 3.0
Y = np.exp(c*t)
w[0] = y_0
for i in range(0, len(t)-1):


ax = fig.add_subplot(1, 2, 2)
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_position(('axes', 0.0))
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
plt.plot(t, w, linewidth = 2)
plt.plot(t, Y, linewidth = 2)
plt.scatter(t, w, s = 50)
plt.xlabel('t', fontsize = 25, loc='right')
plt.ylabel('y', fontsize = 25)
plt.title("h=0.1", fontsize = 25)